In [18]:
import torch
import rasterio
import numpy as np
from torch.utils.data import Dataset
import os
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

In [19]:
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [20]:
root_csv = "../../../data/challenge2026MIASHS/GLC25_PA_metadata_train.csv"
root_images = "../../../data/challenge2026MIASHS/SatelitePatches/PA-train" 



In [21]:
# Chargement du fichier metadata
df = pd.read_csv(root_csv)

print(df.head(10))


        lon        lat  year  geoUncertaintyInM  areaInM2         region  \
0  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
1  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
2  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
3  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
4  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
5  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
6  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
7  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
8  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
9  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   

  country  speciesId  surveyId  
0  France     6874.0       212  
1  France      476.0       212  
2  France    11157.0       212  
3  France     8784.0       212 

In [22]:
# 1. Compter le nombre d'occurrences par espèce
species_counts = df['speciesId'].value_counts()

# 2. Sélectionner les 500 IDs les plus fréquents
top_500_species = species_counts.nlargest(500).index

# 3. Filtrer le DataFrame pour ne garder que ces espèces
df = df[df['speciesId'].isin(top_500_species)].reset_index(drop=True)

print(f"Nombre d'observations après filtrage : {len(df)}")
print(f"Nombre de classes uniques : {len(df['speciesId'].unique())}")

Nombre d'observations après filtrage : 1279346
Nombre de classes uniques : 500


In [23]:
print(df.columns)

Index(['lon', 'lat', 'year', 'geoUncertaintyInM', 'areaInM2', 'region',
       'country', 'speciesId', 'surveyId'],
      dtype='object')


In [24]:
def check_file_exists(row):
    # On utilise la même logique que ton _get_path
    s_id = str(row['surveyId'])
    if len(s_id) >= 4:
        path = os.path.join(root_images, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
    else:
        path = os.path.join(root_images, s_id, f"{s_id}.tiff")
    return os.path.exists(path)

print("Filtrage des données existantes (cela peut prendre un moment)...")
# On crée un masque pour ne garder que les fichiers présents
mask = df.apply(check_file_exists, axis=1)
df = df[mask].reset_index(drop=True)

print(f"Nombre d'images trouvées : {len(df)}")

Filtrage des données existantes (cela peut prendre un moment)...
Nombre d'images trouvées : 1278994


In [25]:
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

# 1. Utilisation des noms exacts 'lat' et 'lon'
coords = df[['lat', 'lon']]
kmeans = KMeans(n_clusters=50, random_state=42, n_init=10).fit(coords)
df['spatial_cluster'] = kmeans.labels_

# 2. Séparation des clusters
unique_clusters = df['spatial_cluster'].unique()
train_clusters, test_clusters = train_test_split(unique_clusters, test_size=0.2, random_state=42)
train_clusters, val_clusters = train_test_split(train_clusters, test_size=0.1, random_state=42)

# 3. Création des DataFrames
train_df = df[df['spatial_cluster'].isin(train_clusters)]
val_df = df[df['spatial_cluster'].isin(val_clusters)]
test_df = df[df['spatial_cluster'].isin(test_clusters)]

print(f"Répartition : Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

Répartition : Train=1097531, Val=48639, Test=132824


In [26]:
train_df['species_label'] = train_df['speciesId'].astype('category').cat.codes

/tmp/ipykernel_2815276/339400487.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['species_label'] = train_df['speciesId'].astype('category').cat.codes


In [27]:
print(train_df.columns)
# Tu dois voir 'species_label' dans la liste. 
# Si ce n'est pas le cas, relance la cellule du split.

Index(['lon', 'lat', 'year', 'geoUncertaintyInM', 'areaInM2', 'region',
       'country', 'speciesId', 'surveyId', 'spatial_cluster', 'species_label'],
      dtype='object')


In [28]:
import torch
import rasterio
import numpy as np
import os
from torch.utils.data import Dataset

class SentinelSpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        """
        Args:
            df (pd.DataFrame): Le DataFrame contenant surveyId et species_label.
            root_dir (str): Chemin vers PA-train (ex: root_images).
            transform (callable, optional): Transformations PyTorch à appliquer.
        """
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _get_path(self, survey_id):
        """
        Applique la règle du challenge :
        surveyId 3018575 -> path: .../75/85/3018575.tiff
        """
        s_id = str(survey_id)
        
        if len(s_id) >= 4:
            # On récupère les deux derniers chiffres (ex: 75)
            part1 = s_id[-2:]
            # On récupère les deux chiffres précédents (ex: 85)
            part2 = s_id[-4:-2]
            return os.path.join(self.root_dir, part1, part2, f"{s_id}.tiff")
        else:
            # Cas des IDs courts (ex: 1 -> ./1/1.tiff)
            return os.path.join(self.root_dir, s_id, f"{s_id}.tiff")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        survey_id = row['surveyId']
        path = self._get_path(survey_id)
        
        try:
            # 1. Lecture du fichier TIFF (4 bandes : R, G, B, NIR)
            with rasterio.open(path) as src:
                # rasterio lit en (Bandes, Hauteur, Largeur) -> (4, 64, 64)
                img = src.read().astype(np.float32)
                
            # 2. Calcul de l'NDVI (Bandes Sentinel-2 dans PA-train : Red=band 1, NIR=band 4)
            # Indice 0: Rouge | Indice 1: Vert | Indice 2: Bleu | Indice 3: NIR
            red = img[0, :, :] 
            nir = img[3, :, :]
            
            # Formule : (NIR - R) / (NIR + R)
            # On ajoute 1e-8 pour éviter la division par zéro
            ndvi = (nir - red) / (nir + red + 1e-8)
            
            # 3. Concaténation pour obtenir 5 canaux (R, G, B, NIR, NDVI)
            # np.newaxis transforme ndvi (64, 64) en (1, 64, 64)
            img_5c = np.concatenate([img, ndvi[np.newaxis, :, :]], axis=0)
            
            # 4. Normalisation
            # Les valeurs Sentinel-2 sont souvent codées sur 10 000 (Réflectance)
            # On ramène tout entre 0 et 1 pour aider le ResNet à converger
            img_5c = img_5c / 10000.0
            
            # 5. Conversion en tenseur PyTorch et application des transformations
            img_tensor = torch.from_numpy(img_5c)
            
            if self.transform:
                img_tensor = self.transform(img_tensor)
                
            return img_tensor, int(row['species_label'])

        except Exception as e:
            # Gestion robuste si un fichier est corrompu ou manquant
            print(f"Erreur de lecture : {path} -> {e}")
            # Retourne un tenseur de zéros pour ne pas casser la boucle d'entraînement
            return torch.zeros((5, 64, 64)), int(row['species_label'])

In [29]:
from torchvision import transforms

# Définition des transformations pour l'entraînement
train_transform = transforms.Compose([
    # 1. Flips (Horizontal et Vertical)
    # Très important en satellite : l'espèce est la même peu importe l'orientation.
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    
    # 2. Rotations aléatoires
    # On autorise toutes les rotations (0 à 180° ou plus)
    transforms.RandomRotation(degrees=180),
    
    # 3. Optionnel : Ajout de bruit Gaussien léger
    # Utile si tu as peu de données pour améliorer la robustesse
    # transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
])

# Pour la validation et le test : AUCUNE transformation géométrique
# On garde les données brutes pour une évaluation juste.
val_transform = None

In [30]:
from torch.utils.data import DataLoader

# 1. Créer les instances de Dataset
train_dataset = SentinelSpeciesDataset(train_df, root_images, transform=train_transform)
val_dataset = SentinelSpeciesDataset(val_df, root_images, transform=None)

# 2. Créer les DataLoaders
# On utilise num_workers=4 pour rester dans tes clous de 25% CPU
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=4
)

In [31]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

def get_species_model(num_classes):
    # Chargement d'un ResNet-50 pré-entraîné
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    
    # Adaptation de la première couche pour 5 canaux (RGB + NIR + NDVI)
    existing_layer = model.conv1
    new_layer = nn.Conv2d(in_channels=5, 
                          out_channels=existing_layer.out_channels, 
                          kernel_size=existing_layer.kernel_size, 
                          stride=existing_layer.stride, 
                          padding=existing_layer.padding, 
                          bias=existing_layer.bias)
    
    # On recopie les poids ImageNet et on initialise les nouveaux canaux
    with torch.no_grad():
        new_layer.weight[:, :3, :, :] = existing_layer.weight
        new_layer.weight[:, 3:, :, :] = existing_layer.weight.mean(dim=1, keepdim=True).repeat(1, 2, 1, 1)
    
    model.conv1 = new_layer
    
    # Adaptation de la sortie au nombre d'espèces du challenge
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

# Initialisation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(df['speciesId'].unique()) # Utilise l'ID unique des espèces
model = get_species_model(num_classes).to(device)

# Optimiseur (AdamW est excellent pour les ResNet) et fonction de perte
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

In [32]:
from sklearn.metrics import f1_score

def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            
            # On stocke tout pour le calcul global
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calcul rigoureux
    avg_loss = val_loss / len(loader)
    # average='micro' renvoie une valeur entre 0 et 1 (ex: 0.025)
    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    
    return avg_loss, micro_f1

In [33]:
# 1. Chargement initial
df = pd.read_csv(root_csv)

# 2. FILTRAGE DES 500 ESPÈCES (À faire en premier !)
species_counts = df['speciesId'].value_counts()
top_500_species = species_counts.nlargest(500).index
df = df[df['speciesId'].isin(top_500_species)].reset_index(drop=True)

# 3. FILTRAGE DES FICHIERS EXISTANTS (On recalcule le masque ici)
print("Vérification des fichiers TIFF sur le disque (Top 500)...")
def check_file_exists(row):
    s_id = str(row['surveyId'])
    if len(s_id) >= 4:
        path = os.path.join(root_images, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
    else:
        path = os.path.join(root_images, s_id, f"{s_id}.tiff")
    return os.path.exists(path)

# On recalcule un masque tout neuf sur le df déjà filtré à 500
new_mask = df.apply(check_file_exists, axis=1)
df = df[new_mask].reset_index(drop=True)

# 4. CRÉATION DES LABELS (0 à 499)
df['species_label'] = df['speciesId'].astype('category').cat.codes

# 5. SPLIT SPATIAL
coords = df[['lat', 'lon']]
kmeans = KMeans(n_clusters=50, random_state=42, n_init=10).fit(coords)
df['spatial_cluster'] = kmeans.labels_

unique_clusters = df['spatial_cluster'].unique()
train_clusters, test_clusters = train_test_split(unique_clusters, test_size=0.2, random_state=42)
train_clusters, val_clusters = train_test_split(train_clusters, test_size=0.1, random_state=42)

train_df = df[df['spatial_cluster'].isin(train_clusters)].copy()
val_df = df[df['spatial_cluster'].isin(val_clusters)].copy()

print(f"Final : {len(df)} images pour {len(df['species_label'].unique())} espèces.")

Vérification des fichiers TIFF sur le disque (Top 500)...
Final : 1278994 images pour 500 espèces.


In [34]:
train_dataset = SentinelSpeciesDataset(train_df, root_images, transform=train_transform)
val_dataset = SentinelSpeciesDataset(val_df, root_images, transform=None)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [35]:
epochs = 5
print(f"{'Epoch':<7} | {'Train Loss':<12} | {'Val Loss':<10} | {'Micro F1':<10} | {'LR':<8}")
print("-" * 60)

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        
        # Petit feedback visuel tous les 500 batches (optionnel)
        if batch_idx % 2500 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
        # Optionnel : Valider plus souvent si l'époque est trop longue
        if batch_idx % 5000 == 0 and batch_idx > 0:
            # Calcul des métriques temporaires
            tmp_train_loss = running_train_loss / (batch_idx + 1)
            v_loss, v_f1 = validate(model, val_loader, criterion, device)
            current_lr = optimizer.param_groups[0]['lr']
            
            # Affichage de la ligne (on ajoute un "+" pour indiquer que c'est en cours d'époque)
            print(f"{epoch+1:<6}+ | {tmp_train_loss:<12.4f} | {v_loss:<10.4f} | {v_f1:<10.4f} | {current_lr:<8.2e}")
            
            model.train() # Retour en mode entraînement

    # Calcul des métriques de fin d'époque
    avg_train_loss = running_train_loss / len(train_loader)
    avg_val_loss, micro_f1 = validate(model, val_loader, criterion, device)
    current_lr = optimizer.param_groups[0]['lr']

    # AFFICHAGE DE TA LIGNE DE DONNÉES
    print(f"{epoch+1:<7} | {avg_train_loss:<12.4f} | {avg_val_loss:<10.4f} | {micro_f1:<10.4f} | {current_lr:<8.2e}")

# Sauvegarde
torch.save(model.state_dict(), "resnet50_species_sentinel2.pth")

Epoch   | Train Loss   | Val Loss   | Micro F1   | LR      
------------------------------------------------------------
Epoch 1 | Batch 0/34298 | Loss: 6.2019
Epoch 1 | Batch 2500/34298 | Loss: 5.1537
Epoch 1 | Batch 5000/34298 | Loss: 4.7900
1     + | 5.3055       | 5.5036     | 0.0275     | 1.00e-04
Epoch 1 | Batch 7500/34298 | Loss: 4.8778
Epoch 1 | Batch 10000/34298 | Loss: 4.5339
1     + | 5.1600       | 5.2983     | 0.0315     | 1.00e-04
Epoch 1 | Batch 12500/34298 | Loss: 4.9258
Epoch 1 | Batch 15000/34298 | Loss: 5.3697
1     + | 5.0783       | 5.3003     | 0.0303     | 1.00e-04
Epoch 1 | Batch 17500/34298 | Loss: 4.9440
Epoch 1 | Batch 20000/34298 | Loss: 4.3869
1     + | 5.0234       | 5.3010     | 0.0309     | 1.00e-04
Epoch 1 | Batch 22500/34298 | Loss: 4.8873
Epoch 1 | Batch 25000/34298 | Loss: 4.7287
1     + | 4.9830       | 5.2356     | 0.0348     | 1.00e-04
Epoch 1 | Batch 27500/34298 | Loss: 4.8622
Epoch 1 | Batch 30000/34298 | Loss: 4.8639
1     + | 4.9516       | 5.

In [36]:
def get_top_and_flops(model, loader, device, n=5):
    model.eval()
    successes = [] # Liste de (image, pred, label, prob)
    failures = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            conf, preds = torch.max(probs, 1)
            
            for i in range(len(preds)):
                img = images[i].cpu()
                p = preds[i].item()
                l = labels[i].item()
                c = conf[i].item()
                
                if p == l:
                    successes.append((img, p, l, c))
                else:
                    failures.append((img, p, l, c))
            
            if len(successes) > 50 and len(failures) > 50: break # On s'arrête quand on en a assez

    # Trier par confiance
    successes.sort(key=lambda x: x[3], reverse=True)
    failures.sort(key=lambda x: x[3], reverse=True) # Flops où le modèle était "sûr de lui"
    
    return successes[:n], failures[:n]

In [39]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# --- Hooks pour Grad-CAM (Ne change pas) ---
activations = []
def hook_activations(module, input, output):
    activations.append(output)

gradients = []
def hook_gradients(module, grad_input, grad_output):
    gradients.append(grad_output[0])

def setup_gradcam(model):
    target_layer = model.layer4
    target_layer.register_forward_hook(hook_activations)
    target_layer.register_full_backward_hook(hook_gradients)

def generate_gradcam(model, input_tensor, target_class_index, device):
    model.eval()
    activations.clear()
    gradients.clear()
    
    input_tensor = input_tensor.to(device)
    output = model(input_tensor)
    
    one_hot_output = torch.zeros_like(output)
    one_hot_output[0][target_class_index] = 1
    
    model.zero_grad()
    output.backward(gradient=one_hot_output, retain_graph=True)
    
    grads = gradients[0].cpu().data.numpy()[0]
    acts = activations[0].cpu().data.numpy()[0]
    
    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * acts[i, :, :]
    
    cam = np.maximum(cam, 0)
    cam = cam / (np.max(cam) + 1e-8)
    return cam # On retourne la petite carte (ex: 2x2 ou 7x7)

def plot_results_gallery(samples, title="Galerie"):
    plt.figure(figsize=(20, 6))
    for i, (img, pred, label, conf) in enumerate(samples):
        # 1. Générer la petite Heatmap
        cam = generate_gradcam(model, img.unsqueeze(0), pred, device)
        
        # 2. Récupérer le canal NDVI pour le fond
        ndvi = img[4].numpy()
        
        # 3. Affichage avec Matplotlib (il gère l'agrandissement tout seul !)
        plt.subplot(1, len(samples), i + 1)
        
        # On affiche le fond
        plt.imshow(ndvi, cmap='gray')
        
        # On superpose la heatmap ('jet' = rouge/bleu, alpha = transparence)
        # extent=[0,63,63,0] force la petite heatmap à s'étirer sur toute l'image 64x64
        plt.imshow(cam, cmap='jet', alpha=0.5, extent=[0,63,63,0], interpolation='bilinear')
        
        plt.title(f"Prédit: {pred}\nRéel: {label}\nConf: {conf:.2%}")
        plt.axis('off')
    plt.suptitle(title, fontsize=16)
    plt.show()

In [41]:
def plot_results_gallery(samples, title="Galerie"):
    if not samples:
        print(f"Aucun échantillon trouvé pour : {title}")
        return

    plt.figure(figsize=(20, 6))
    for i, (img, pred, label, conf) in enumerate(samples):
        # 1. Générer la heatmap Grad-CAM (la petite matrice d'activation)
        cam = generate_gradcam(model, img.unsqueeze(0), pred, device)
        
        # 2. Récupérer le canal NDVI (indice 4) pour servir de fond
        # On le normalise entre 0 et 1 pour l'affichage
        ndvi = img[4].numpy()
        ndvi_min, ndvi_max = ndvi.min(), ndvi.max()
        ndvi_norm = (ndvi - ndvi_min) / (ndvi_max - ndvi_min + 1e-8)
        
        # 3. Affichage
        plt.subplot(1, len(samples), i + 1)
        
        # On affiche l'image NDVI en fond (niveaux de gris)
        plt.imshow(ndvi_norm, cmap='gray', interpolation='nearest')
        
        # On superpose la heatmap Grad-CAM
        # cmap='jet' simule les couleurs thermiques (rouge = important)
        # extent=[0,63,63,0] étire la petite heatmap sur les 64x64 pixels
        # alpha=0.5 permet de voir le NDVI par transparence
        plt.imshow(cam, cmap='jet', alpha=0.5, extent=[0, 63, 63, 0], interpolation='bilinear')
        
        plt.title(f"Prédit: {pred}\nRéel: {label}\nConf: {conf:.2%}", fontsize=10)
        plt.axis('off')
        
    plt.suptitle(title, fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

In [42]:
class SentinelTestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _get_path(self, survey_id):
        s_id = str(survey_id)
        if len(s_id) >= 4:
            return os.path.join(self.root_dir, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
        return os.path.join(self.root_dir, s_id, f"{s_id}.tiff")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        survey_id = row['surveyId']
        path = self._get_path(survey_id)
        
        try:
            with rasterio.open(path) as src:
                img = src.read().astype(np.float32)
            
            # Calcul NDVI identique au train
            ndvi = (img[3] - img[0]) / (img[3] + img[0] + 1e-8)
            img_5c = np.concatenate([img, ndvi[np.newaxis, :, :]], axis=0)
            img_5c = img_5c / 10000.0
            
            return torch.from_numpy(img_5c), survey_id
        except:
            # Si image manquante en test, on renvoie du vide pour ne pas planter
            return torch.zeros((5, 64, 64)), survey_id

In [43]:
root_test_csv = "../../../data/challenge2026MIASHS/GLC25_PA_metadata_test.csv"
root_test_images = "../../../data/challenge2026MIASHS/SatelitePatches/PA-test"

df_test = pd.read_csv(root_test_csv)
test_dataset = SentinelTestDataset(df_test, root_test_images)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

# On prépare le dictionnaire de traduction (Index -> ID réel)
# ATTENTION : Utilise bien le 'df' d'entraînement pour avoir toutes les catégories !
mapping = dict(enumerate(df['speciesId'].astype('category').cat.categories))

In [44]:
def generate_submission(model, test_loader, device, mapping, filename="submission.csv"):
    model.eval()
    results = []

    print("Génération des prédictions pour le fichier de soumission...")
    with torch.no_grad():
        for images, survey_ids in test_loader:
            images = images.to(device)
            outputs = model(images)
            
            # On récupère l'index de l'espèce la plus probable
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(preds)):
                # Traduction de l'index en speciesId réel
                species_id_real = mapping[preds[i].item()]
                # Formatage : "surveyId,prediction"
                results.append({
                    'surveyId': survey_ids[i].item(),
                    'predictions': str(species_id_real)
                })

    # Création du DataFrame final
    submission_df = pd.DataFrame(results)
    submission_df.to_csv(filename, index=False)
    print(f"Fichier sauvegardé sous : {filename}")

# --- LANCEMENT ---
# Note : Ton test_loader doit renvoyer (image, surveyId) et non (image, label)
generate_submission(model, test_loader, device, mapping)

Génération des prédictions pour le fichier de soumission...
Fichier sauvegardé sous : submission.csv
